In [1]:

import os, time, math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set, Optional

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


SEED = 42
VAL_FRAC = 0.05

MAX_LEN = 140                  
MAX_RAW_LEN = MAX_LEN - 2      

LIMIT_TRAIN = None             
LIMIT_VAL   = None
LIMIT_TEST  = 200_000         

CACHE_DIR = "./cache_moses_full"
CACHE_PATH = f"{CACHE_DIR}/moses_full_splits_vocab.pt"

BEST_Z_DIM = 32
BEST_FREE_NATS = 0.05
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-2

N_SAMPLES = 5000
TEMP = 0.9
SAMPLE_BS = 512

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs("./checkpoints_moses_full", exist_ok=True)


ds = load_dataset("katielink/moses")
train_split = ds["train"]
test_split  = ds["test"]

sm_col = "SMILES" if "SMILES" in train_split.column_names else "smiles"
print("SMILES column:", sm_col)
print("HF sizes:", len(train_split), len(test_split))

split = train_split.train_test_split(test_size=VAL_FRAC, seed=SEED, shuffle=True)
train_hf = split["train"]
val_hf   = split["test"]
test_hf  = test_split

print("Split sizes:", len(train_hf), len(val_hf), len(test_hf))



Device: cuda
SMILES column: SMILES
HF sizes: 1584663 352299
Split sizes: 1505429 79234 352299


In [2]:

def canonical(smiles: str) -> Optional[str]:
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m, canonical=True)

def clean_split(hf_ds, limit: Optional[int], desc: str) -> List[str]:
    if limit is not None:
        hf_ds = hf_ds.select(range(min(limit, len(hf_ds))))
    out = []
    for r in tqdm(hf_ds, desc=desc, total=len(hf_ds)):
        s = r[sm_col]
        if not isinstance(s, str):
            continue
        cs = canonical(s)
        if cs is None:
            continue
        if len(cs) > MAX_RAW_LEN:
            continue
        out.append(cs)
    return out

train_sm = clean_split(train_hf, LIMIT_TRAIN, "clean:train_full")
val_sm   = clean_split(val_hf,   LIMIT_VAL,   "clean:val_full")
test_sm  = clean_split(test_hf,  LIMIT_TEST,  "clean:test")

print("Cleaned sizes:", len(train_sm), len(val_sm), len(test_sm))


PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"

chars = set()
for s in train_sm:
    chars.update(s)

itos = [PAD, BOS, EOS] + sorted(chars)
stoi = {c:i for i,c in enumerate(itos)}

pad_id = stoi[PAD]
bos_id = stoi[BOS]
eos_id = stoi[EOS]
vocab_size = len(itos)

print("Vocab size:", vocab_size)
print("Char examples:", itos[:10], "...", itos[-10:])


torch.save({
    "splits": (train_sm, val_sm, test_sm),
    "stoi": stoi,
    "itos": itos,
    "cfg": {
        "dataset": "katielink/moses",
        "MAX_LEN": MAX_LEN,
        "MAX_RAW_LEN": MAX_RAW_LEN,
        "val_fraction": VAL_FRAC,
        "seed": SEED,
        "limits": {"train": LIMIT_TRAIN, "val": LIMIT_VAL, "test": LIMIT_TEST},
        "smiles_col": sm_col,
        "full_train": True
    }
}, CACHE_PATH)

print("Saved cache:", CACHE_PATH)



clean:test: 100%|██████████| 200000/200000 [00:52<00:00, 3839.93it/s]


Cleaned sizes: 1505429 79234 200000
Vocab size: 29
Char examples: ['<PAD>', '<BOS>', '<EOS>', '#', '(', ')', '-', '1', '2', '3'] ... ['O', 'S', '[', ']', 'c', 'l', 'n', 'o', 'r', 's']
Saved cache: ./cache_moses_full/moses_full_splits_vocab.pt


In [3]:

def encode(sm: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    ids = [stoi[BOS]] + [stoi[c] for c in sm if c in stoi] + [stoi[EOS]]
    if len(ids) < max_len:
        ids = ids + [stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = stoi[EOS]
    return ids

class SmilesVAEDataset(Dataset):
    def __init__(self, smiles_list: List[str], stoi: Dict[str,int], max_len: int):
        self.smiles = smiles_list
        self.stoi = stoi
        self.max_len = max_len
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        ids = encode(self.smiles[idx], self.stoi, self.max_len)
        return torch.tensor(ids, dtype=torch.long)

train_ds = SmilesVAEDataset(train_sm, stoi, MAX_LEN)
val_ds   = SmilesVAEDataset(val_sm,   stoi, MAX_LEN)

BATCH_SIZE = 256 if DEVICE == "cuda" else 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          pin_memory=(DEVICE=="cuda"), drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          pin_memory=(DEVICE=="cuda"))

print("Batches:", len(train_loader), len(val_loader))

Batches: 5880 310


In [4]:

@dataclass
class VAEConfig:
    vocab_size: int
    emb_dim: int = 256
    enc_hidden: int = 512
    dec_hidden: int = 512
    num_layers: int = 1
    z_dim: int = 32
    dropout: float = 0.0

class SmilesVAE(nn.Module):
    def __init__(self, cfg: VAEConfig, pad_id: int, bos_id: int, eos_id: int):
        super().__init__()
        self.cfg = cfg
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id

        self.embed = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=pad_id)

        self.enc_gru = nn.GRU(cfg.emb_dim, cfg.enc_hidden, num_layers=cfg.num_layers,
                              batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)
        self.to_mu = nn.Linear(cfg.enc_hidden, cfg.z_dim)
        self.to_logvar = nn.Linear(cfg.enc_hidden, cfg.z_dim)

        self.z_to_h0 = nn.Linear(cfg.z_dim, cfg.dec_hidden * cfg.num_layers)

        self.dec_gru = nn.GRU(cfg.emb_dim, cfg.dec_hidden, num_layers=cfg.num_layers,
                              batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)
        self.fc_out = nn.Linear(cfg.dec_hidden, cfg.vocab_size)

    def encode(self, x):
        emb = self.embed(x)
        _, h = self.enc_gru(emb)
        h_last = h[-1]
        return self.to_mu(h_last), self.to_logvar(h_last)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)

        x_in = x[:, :-1]
        x_tgt = x[:, 1:]

        B = z.size(0)
        L = self.cfg.num_layers
        h0 = self.z_to_h0(z).view(L, B, self.cfg.dec_hidden).contiguous()

        emb = self.embed(x_in)
        out, _ = self.dec_gru(emb, h0)
        logits = self.fc_out(out)
        return logits, x_tgt, mu, logvar

def recon_loss(logits, targets, pad_id: int):
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B*T, V), targets.reshape(B*T), ignore_index=pad_id)

In [5]:

def run_epoch_freebits(
    vae: SmilesVAE,
    loader,
    optimizer=None,
    kl_weight: float = 1.0,
    free_nats: float = 0.05
):
    train = optimizer is not None
    vae.train(train)
    total = {"loss": 0.0, "recon": 0.0, "kl_fb": 0.0, "kl_raw": 0.0, "kl_raw_dim": 0.0}
    n = 0

    for x in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits, tgt, mu, logvar = vae(x)
        r = recon_loss(logits, tgt, pad_id)

        kl_per_dim = 0.5 * (torch.exp(logvar) + mu**2 - 1.0 - logvar)
        kl_raw = torch.sum(kl_per_dim, dim=1).mean()
        kl_fb  = torch.sum(torch.clamp(kl_per_dim, min=free_nats), dim=1).mean()

        loss = r + kl_weight * kl_fb

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()

        total["loss"] += float(loss.item())
        total["recon"] += float(r.item())
        total["kl_fb"] += float(kl_fb.item())
        total["kl_raw"] += float(kl_raw.item())
        total["kl_raw_dim"] += float(kl_per_dim.mean().item())
        n += 1

    for k in total:
        total[k] /= max(1, n)
    return total

In [6]:

@torch.no_grad()
def sample_vae(vae: SmilesVAE, n: int, max_len: int, temperature: float = 0.9, batch_size: int = 512) -> List[str]:
    vae.eval()
    samples = []
    remaining = n

    while remaining > 0:
        B = min(batch_size, remaining)
        remaining -= B

        z = torch.randn(B, vae.cfg.z_dim, device=DEVICE)
        L = vae.cfg.num_layers
        h = vae.z_to_h0(z).view(L, B, vae.cfg.dec_hidden).contiguous()

        cur = torch.full((B, 1), bos_id, dtype=torch.long, device=DEVICE)
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        out_ids = [[] for _ in range(B)]

        for _ in range(max_len - 1):
            emb = vae.embed(cur)
            out, h = vae.dec_gru(emb, h)
            logits = vae.fc_out(out[:, -1, :]) / max(1e-8, temperature)
            probs = F.softmax(logits, dim=-1)

            nxt = torch.multinomial(probs, 1).squeeze(1)
            nxt = torch.where(finished, torch.tensor(eos_id, device=DEVICE), nxt)

            nxt_cpu = nxt.tolist()
            fin_cpu = finished.tolist()
            for i, nid in enumerate(nxt_cpu):
                if fin_cpu[i]:
                    continue
                if nid == eos_id:
                    finished[i] = True
                elif nid not in (pad_id, bos_id):
                    out_ids[i].append(nid)

            cur = nxt.unsqueeze(1)
            if finished.all():
                break

        samples.extend(["".join(itos[j] for j in ids) for ids in out_ids])

    return samples

In [7]:

def to_mol(smiles: str):
    return Chem.MolFromSmiles(smiles)

def canonical_eval(smiles: str) -> Optional[str]:
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m, canonical=True)

train_canon: Set[str] = set()
for s in train_sm:
    cs = canonical_eval(s)
    if cs is not None:
        train_canon.add(cs)

def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    valid_canon = []
    for s in smiles_list:
        cs = canonical_eval(s)
        if cs is not None:
            valid_canon.append(cs)
    return len(valid_canon) / max(1, len(smiles_list)), valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    return 0.0 if len(valid_canon)==0 else len(set(valid_canon))/len(valid_canon)

def novelty(valid_canon: List[str], train_set: Set[str]) -> float:
    return 0.0 if len(valid_canon)==0 else sum(1 for s in valid_canon if s not in train_set)/len(valid_canon)

def novelty_unique(valid_canon: List[str], train_set: Set[str]) -> float:
    uv = set(valid_canon)
    return 0.0 if len(uv)==0 else sum(1 for s in uv if s not in train_set)/len(uv)

def compute_properties(valid_canon: List[str]) -> Dict[str,float]:
    if len(valid_canon)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    mws, logps, qeds = [], [], []
    for s in valid_canon:
        m = to_mol(s)
        if m is None:
            continue
        mws.append(Descriptors.MolWt(m))
        logps.append(Crippen.MolLogP(m))
        qeds.append(QED.qed(m))
    if len(mws)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    return {"mw_mean": sum(mws)/len(mws), "logp_mean": sum(logps)/len(logps), "qed_mean": sum(qeds)/len(qeds)}

def evaluate_smiles(samples: List[str], train_set: Set[str]) -> Dict[str,float]:
    v, valid_canon = validity(samples)
    out = {
        "n_samples": len(samples),
        "validity": v,
        "n_valid": len(valid_canon),
        "uniqueness": uniqueness(valid_canon),
        "novelty": novelty(valid_canon, train_set),
        "novelty_unique": novelty_unique(valid_canon, train_set),
    }
    out.update(compute_properties(valid_canon))
    return out


cfg = VAEConfig(vocab_size=vocab_size, z_dim=BEST_Z_DIM)
vae = SmilesVAE(cfg, pad_id, bos_id, eos_id).to(DEVICE)
opt = torch.optim.AdamW(vae.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
print("\n" + "="*80)
print(f"Training FULL-MOSES FreeBits VAE | z_dim={BEST_Z_DIM} | free_nats={BEST_FREE_NATS} | epochs={EPOCHS}")

for epoch in range(1, EPOCHS+1):
    kl_w = min(1.0, epoch / EPOCHS)  # warmup over all epochs
    t0 = time.time()
    tr = run_epoch_freebits(vae, train_loader, optimizer=opt, kl_weight=kl_w, free_nats=BEST_FREE_NATS)
    va = run_epoch_freebits(vae, val_loader, optimizer=None, kl_weight=kl_w, free_nats=BEST_FREE_NATS)
    t1 = time.time()

    print(f"Epoch {epoch:02d} | kl_w={kl_w:.2f} | "
          f"train L={tr['loss']:.3f} (R={tr['recon']:.3f}, KLraw={tr['kl_raw']:.3f}, KLfb={tr['kl_fb']:.3f}) | "
          f"val L={va['loss']:.3f} (R={va['recon']:.3f}, KLraw={va['kl_raw']:.3f}, KLfb={va['kl_fb']:.3f}) | {t1-t0:.1f}s")

    history.append({
        "epoch": epoch, "kl_w": kl_w,
        **{f"train_{k}": v for k,v in tr.items()},
        **{f"val_{k}": v for k,v in va.items()},
    })

hist_df = pd.DataFrame(history)
hist_csv = f"./checkpoints_moses_full/freebits_fullmoses_z{BEST_Z_DIM}_fn{BEST_FREE_NATS}_history.csv"
hist_df.to_csv(hist_csv, index=False)
print("Saved history:", hist_csv)


t0 = time.time()
gen = sample_vae(vae, n=N_SAMPLES, max_len=MAX_LEN, temperature=TEMP, batch_size=SAMPLE_BS)
t1 = time.time()

m = evaluate_smiles(gen, train_canon)
gen_seconds = t1 - t0
m["gen_seconds"] = gen_seconds
m["samples_per_sec"] = N_SAMPLES / gen_seconds
m["valid_per_sec"] = m["n_valid"] / gen_seconds

m["z_dim"] = BEST_Z_DIM
m["free_nats"] = BEST_FREE_NATS
m["final_val_recon"] = history[-1]["val_recon"]
m["final_val_kl_raw"] = history[-1]["val_kl_raw"]
m["final_val_kl_fb"] = history[-1]["val_kl_fb"]
m["final_val_kl_raw_dim"] = history[-1]["val_kl_raw_dim"]

print("\nEval metrics (FULL train novelty reference):")
print(m)

ckpt_path = f"./checkpoints_moses_full/freebits_fullmoses_z{BEST_Z_DIM}_fn{BEST_FREE_NATS}.pt"
torch.save({
    "model_state": vae.state_dict(),
    "cfg": cfg.__dict__,
    "stoi": stoi,
    "itos": itos,
    "max_len": MAX_LEN
}, ckpt_path)
print("Saved checkpoint:", ckpt_path)


Training FULL-MOSES FreeBits VAE | z_dim=32 | free_nats=0.05 | epochs=10
Epoch 01 | kl_w=0.10 | train L=0.819 (R=0.659, KLraw=0.644, KLfb=1.600) | val L=0.741 (R=0.581, KLraw=0.575, KLfb=1.600) | 271.4s
Epoch 02 | kl_w=0.20 | train L=0.885 (R=0.565, KLraw=0.542, KLfb=1.600) | val L=0.874 (R=0.554, KLraw=0.625, KLfb=1.600) | 272.7s
Epoch 03 | kl_w=0.30 | train L=1.025 (R=0.545, KLraw=0.491, KLfb=1.600) | val L=1.019 (R=0.539, KLraw=0.494, KLfb=1.600) | 273.8s
Epoch 04 | kl_w=0.40 | train L=1.174 (R=0.534, KLraw=0.589, KLfb=1.600) | val L=1.171 (R=0.531, KLraw=0.991, KLfb=1.600) | 272.9s
Epoch 05 | kl_w=0.50 | train L=1.327 (R=0.527, KLraw=0.908, KLfb=1.600) | val L=1.324 (R=0.524, KLraw=1.053, KLfb=1.600) | 272.3s
Epoch 06 | kl_w=0.60 | train L=1.481 (R=0.521, KLraw=0.995, KLfb=1.600) | val L=1.481 (R=0.521, KLraw=1.132, KLfb=1.600) | 272.3s
Epoch 07 | kl_w=0.70 | train L=1.637 (R=0.517, KLraw=0.980, KLfb=1.600) | val L=1.638 (R=0.518, KLraw=1.030, KLfb=1.600) | 272.3s
Epoch 08 | kl_w=